# 06 - Single & Batch Download (Index / RGB GeoTIFF)

**What:** Download a vegetation-index or RGB Sentinel-2 GeoTIFF for a single date, batch-download the same product across every selected date, and pre-select specific dates before downloading. The AOI clip is taken on the *buffered* geometry exactly as the plugin does.

**Why:** Agronomists need the raw clipped raster (not just an on-screen layer) to take into other tools. The plugin lets them grab one date, or fan out across all dates in the filtered time series in one run, optionally trimming the date list with a scrollable checklist first.

**Legacy reference (`legacy:ravi_dialog.py`):**
- `load_index()` (~2312) and `load_rgb()` (~2533) — the clip + `getDownloadURL(...)` paths at **10 m scale**, GeoTIFF, `EPSG:4326`.
- `index_batch_clicked()` / `rgb_batch_clicked()` (~610-662) — loop over `recorte_datas` (or `df_aux.date` when `None`), driving `load_index` / `load_rgb` per date.
- `datasrecorte_clicked()` / `apply_changes()` / `recorte_datas` (~1422-1592) — per-date selection; `recorte_datas` is the resulting list of `YYYY-MM-DD` strings.
- `apply_buffer()` (~2483) — buffer the AOI by the slider value (meters) before clipping.
- Collection build (~3628-3633) — `COPERNICUS/S2_SR_HARMONIZED`, `date` property via `image.date().format('YYYY-MM-dd')`.
- `calculate_vegetation_index()` (~2284) — `NDVI = normalizedDifference(['B8','B4'])` renamed to `index`.

In [ ]:
# --- Setup ---
import os
import ee
import pandas as pd
import requests

# Service-account auth via the GEE env var (same as dev/testing.ipynb).
credentials = ee.ServiceAccountCredentials(None, os.environ["GEE"])
ee.Initialize(credentials)
print("Earth Engine initialized")

In [ ]:
# --- AOI + dates + masked S2 collection with an NDVI band ---
# AOI loaded from the project shapefile (same area as dev/testing.ipynb).
# In the plugin this is a FeatureCollection (self.aoi); the loader returns one so
# apply_buffer's feature.buffer(...) mapping works the same way as legacy.
import zipfile

import geopandas as gpd


def load_aoi_from_shapefile(shapefile_path):
    """Load an AOI from a (zipped) shapefile as an ee.FeatureCollection.

    Same loader as dev/testing.ipynb: reproject to EPSG:4326, dissolve to a
    single geometry, strip any Z coordinate, wrap in a FeatureCollection.
    """
    if shapefile_path.endswith(".zip"):
        with zipfile.ZipFile(shapefile_path, "r") as zip_ref:
            shapefile_within_zip = None
            for file in zip_ref.namelist():
                if file.lower().endswith(".shp"):
                    shapefile_within_zip = file
                    break
            if not shapefile_within_zip:
                raise FileNotFoundError(f"No .shp file found inside the zip archive: {shapefile_path}")
            gpd_aoi = gpd.read_file(f"zip://{shapefile_path}/{shapefile_within_zip}")
    else:
        gpd_aoi = gpd.read_file(shapefile_path)

    gpd_aoi = gpd_aoi.to_crs(epsg=4326)

    if gpd_aoi.empty:
        raise ValueError(f"The shapefile at {shapefile_path} does not contain any geometries.")

    if len(gpd_aoi) > 1:
        gpd_aoi = gpd_aoi.dissolve()

    geometry = gpd_aoi.geometry.iloc[0]
    geojson = geometry.__geo_interface__

    if geojson["type"] == "Polygon":
        geojson["coordinates"] = [
            list(map(lambda coord: coord[:2], ring)) for ring in geojson["coordinates"]
        ]
    elif geojson["type"] == "MultiPolygon":
        geojson["coordinates"] = [
            [list(map(lambda coord: coord[:2], ring)) for ring in polygon]
            for polygon in geojson["coordinates"]
        ]

    ee_geometry = ee.Geometry(geojson)
    feature = ee.Feature(ee_geometry)
    return ee.FeatureCollection([feature])


aoi = load_aoi_from_shapefile("contorno_area_total.zip")
aoi_geom = aoi.geometry()

START_DATE = "2023-06-01"
END_DATE = "2023-08-31"
CLOUD_PERCENT = 60  # legacy: ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', nuvem)

# Build the collection exactly as legacy (ravi_dialog ~3628):
#   - S2_SR_HARMONIZED, filtered by date + bounds
#   - a 'date' string property: image.date().format('YYYY-MM-dd')
#   - tile-level cloud percentage filter
sentinel2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate(START_DATE, END_DATE)
    .filterBounds(aoi)
    .map(lambda image: image.set("date", image.date().format("YYYY-MM-dd")))
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUD_PERCENT))
)

# Cloud/shadow mask via the Scene Classification Layer (legacy SCL_mask, ~3832).
# SCL classes excluded: 3=cloud shadow, 8=cloud medium prob, 9=cloud high prob,
# 10=thin cirrus, 11=snow/ice.
SCL_EXCLUDE = [3, 8, 9, 10, 11]


def mask_cloud_and_shadows(image):
    scl = image.select("SCL")
    mask = ee.Image.constant(1)
    for class_value in SCL_EXCLUDE:
        mask = mask.And(scl.neq(class_value))
    return image.updateMask(mask)


sentinel2 = sentinel2.map(mask_cloud_and_shadows)


# NDVI band, legacy calculate_vegetation_index (~2284): normalizedDifference(['B8','B4'])
# renamed to 'index'. Added as a band so single/batch can clip+download the index image.
def add_ndvi(image):
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("index")
    return image.addBands(ndvi)


sentinel2 = sentinel2.map(add_ndvi)

print("Collection size:", sentinel2.size().getInfo())

In [ ]:
# --- Available dates (per-date selection source) ---
# In the plugin, dataunica dropdown / df_aux.date provide the list of one-per-day
# acquisition dates. Here we pull the distinct 'date' property as plain strings.
# This is the universe of dates the selection (recorte_datas) is drawn from.
available_dates = sorted(set(sentinel2.aggregate_array("date").getInfo()))
print(f"{len(available_dates)} available dates:")
for d in available_dates:
    print(" ", d)

In [ ]:
# --- Per-date selection (legacy: datasrecorte_clicked + apply_changes -> recorte_datas) ---
# The plugin pops a scrollable checklist. Apply sets:
#     self.recorte_datas = [cb.text() for cb, _, _ in self.checkboxes if cb.isChecked()]
# i.e. a list of 'YYYY-MM-DD' strings. When None, the batch falls back to ALL dates
# (df_aux.date.tolist()). We emulate the selection here by picking a subset.

# recorte_datas = None  -> means "use every available date" (legacy default)
recorte_datas = available_dates[:3] if len(available_dates) >= 3 else available_dates

# Resolve the effective list of dates the batch will iterate over (legacy logic):
selected_dates = available_dates if recorte_datas is None else recorte_datas
print("Selected dates for download:", selected_dates)

In [ ]:
# --- apply_buffer (legacy ~2483) ---
# Buffer every feature of the AOI FeatureCollection by the slider value (meters).
# Positive grows, negative shrinks; 0 leaves it untouched. The buffered AOI is what
# every clip / getDownloadURL region uses downstream.
BUFFER_DISTANCE = 0  # horizontalSlider_buffer.value(), in meters


def apply_buffer(aoi, buffer_distance=BUFFER_DISTANCE):
    """Applies a buffer to the AOI geometry (legacy apply_buffer)."""
    if buffer_distance != 0:
        print(f"Buffer distance: {buffer_distance} meters")
        return aoi.map(lambda feature: feature.buffer(buffer_distance))
    else:
        print("No buffer applied")
        return aoi


buffered_aoi = apply_buffer(aoi)

In [ ]:
# --- Single-date download (legacy load_index ~2312) ---
# Pick one date, grab the first image for it, clip the index band to the buffered AOI,
# and build the download URL at 10 m scale / GeoTIFF / EPSG:4326 (verbatim params).
VEGETATION_INDEX = "NDVI"
single_date = selected_dates[0]

first_image = sentinel2.filter(ee.Filter.inList("date", [single_date])).first()
first_image = first_image.clip(buffered_aoi)

# 'index' is the NDVI band added in cell 3 (calculate_vegetation_index renames to 'index').
index_image = first_image.select("index")

url = index_image.getDownloadURL(
    {
        "scale": 10,
        "region": buffered_aoi.geometry().bounds().getInfo(),
        "format": "GeoTIFF",
        "crs": "EPSG:4326",  # Use WGS84 for compatibility
    }
)
output_file = f"{VEGETATION_INDEX}_{single_date}.tiff"
print("Download URL:", url)
print("Would write to:", output_file)

# Optional actual fetch (legacy requests.get path). Guarded: only run on a small AOI.
WRITE_FILE = False
if WRITE_FILE:
    response = requests.get(url)
    if response.status_code == 200:
        with open(output_file, "wb") as f:
            f.write(response.content)
        print(f"{VEGETATION_INDEX} image downloaded as {output_file}")
    else:
        print(f"Failed to download image. HTTP Status: {response.status_code}")

In [ ]:
# --- Batch download (legacy index_batch_clicked / rgb_batch_clicked ~610-662) ---
# Legacy loops the dates (recorte_datas, else df_aux.date) and calls load_index /
# load_rgb per date. Here we build one download URL per selected date and collect
# them. We DON'T fetch every GeoTIFF (that's the slow, QGIS-freezing part) -- this
# stays lightweight and runnable; flip WRITE_BATCH to actually download.
PRODUCT = "index"  # 'index' (NDVI) or 'rgb'
RGB_BANDS = ["B4", "B3", "B2"]  # load_rgb selects all bands; R/G/B shown here
region = buffered_aoi.geometry().bounds().getInfo()

batch = []
for data in selected_dates:
    img = sentinel2.filter(ee.Filter.inList("date", [data])).first().clip(buffered_aoi)
    if PRODUCT == "rgb":
        img = img.select(RGB_BANDS)
        fname = f"Sentinel2_RGB_{data}.tiff"
    else:
        img = img.select("index")
        fname = f"{VEGETATION_INDEX}_{data}.tiff"
    url = img.getDownloadURL(
        {
            "scale": 10,
            "region": region,
            "format": "GeoTIFF",
            "crs": "EPSG:4326",
        }
    )
    print(f"Built URL for date: {data} -> {fname}")
    batch.append({"date": data, "filename": fname, "url": url})

batch_df = pd.DataFrame(batch)
print(f"\n{len(batch_df)} download URLs prepared.")
batch_df[["date", "filename"]]

In [ ]:
# --- Optional: actually fetch the batch (legacy requests.get per date) ---
# Off by default. Each row is one GeoTIFF written to disk; only enable on a small AOI.
WRITE_BATCH = False
if WRITE_BATCH:
    for row in batch:
        resp = requests.get(row["url"])
        if resp.status_code == 200:
            with open(row["filename"], "wb") as f:
                f.write(resp.content)
            print(f"Downloaded {row['filename']}")
        else:
            print(f"Failed {row['filename']}: HTTP {resp.status_code}")
else:
    print("WRITE_BATCH is False -- URLs prepared but no files fetched.")